# 【学習用・解説付き写し】[LB 0.76704] No-Train Anatomy-Calibrated DLTrack- **コンペ**: [UMUD Challenge: Muscle Architecture in Ultrasound Data](https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data)（開催中・残り約3ヶ月 / 165チーム / 賞金3,000 CHF）- **原著者**: phuongncn- **元notebook**: https://www.kaggle.com/code/phuongncn/lb-0-76704-no-train-anatomy-calibrated-dltrack- **Public Score**: 0.76704（**低いほど良い**指標。本日時点の公開notebook最良）- **学習**: なし / **GPU**: なし / **実行時間**: 25秒- **ライセンス**: notebookコードは GPL-3.0## 手法の概要（1段落）超音波画像から筋の**羽状角(PA)・筋束長(FL)・筋厚(MT)**を推定するコンペです。このnotebookは**モデルを一切学習しません**。代わりに、既に採点済みの予測CSV（"C130 anchor"）をbase64+gzipで**ソースコード内に埋め込み**、それを起点に「MTを一律 -0.4mm 補正 → 比例補正との包絡（envelope）を取る →下側の値だけ半歩追加補正 → `FL·sin(PA) ≥ MT` という**解剖学的整合性チェック**をブール値のゲートとして使う」という決定論的な後処理チェーンを適用して `submission.csv` を再構築します。全処理は `Decimal`（60桁精度）で行われ、出力のSHA-256が期待値と一致しない限り**失敗して止まる**設計です。> 📌 これは**学習目的の解説付き写し**です。原著者のコードは変更しておらず、各コードセルの直前に> 日本語の解説Markdownセルを挿入しています。> ⚠️ **重要な但し書き**: このnotebookは「機械学習のコードとして真似すべきお手本」ではありません。> 学習も推論も無く、実質的には**採点済みCSVに手作業で見つけた補正ルールを適用しているだけ**です。> それでも今日これを選んだのは、(1) 現時点でこのコンペの公開最良スコアであり、> (2) **後処理・キャリブレーションだけでスコアがどこまで動くか**を定量的に記録した稀な資料であり、> (3) 同時に **「Public LBを見ながらルールを足していく」ことの危うさ**を考える最良の教材だからです。> 「スコアが高い＝良い手法」と読み替えないための練習として読んでください。

## 評価指標（このコンペで何が測られているか）**タスク**: 骨格筋の超音波画像1枚ごとに、次の3つの数値を**回帰**で予測します。| 変数 | 意味 | 単位 ||---|---|---|| **PA**（Pennation Angle / 羽状角） | 筋束（fascicle）と深部腱膜（aponeurosis）のなす角 | 度 || **FL**（Fascicle Length / 筋束長） | 浅部腱膜と深部腱膜の間の筋束の長さ | mm || **MT**（Muscle Thickness / 筋厚） | 浅部腱膜と深部腱膜の垂直距離 | mm |提出は `image_id, pa_deg, fl_mm, mt_mm` の4列CSV（テスト309行）。テスト画像には**動画から連続5フレームを切り出したもの**が含まれ、時間的な安定性も間接的に問われます。**評価指標: UMUD Score**= 3変数それぞれの **MAE（平均絶対誤差）を、変数ごとに定められた許容値（tolerance）で正規化して合成**したもの。**小さいほど良い**（本notebookは0.76704）。**なぜこの指標か**:- PAは「度」、FLとMTは「mm」で、しかもFL（数十mm）とMT（十数mm〜二十数mm）でも典型的な大きさが違います。  生のMAEを単純合計すると、**数値の大きいFLの誤差だけで指標が決まってしまいます**。  そこで各変数を「臨床的に許容できる誤差幅」で割ってから足すことで、3変数の寄与を揃えています。- MAE（絶対誤差）でありRMSEでないのは、**外れ値に引きずられないため**です。  超音波画像は画質のばらつきが大きく、一部の画像で大きく外すことは避けられません。  RMSEだと数枚の失敗が指標を支配してしまい、「全体的にどれくらい正確か」が見えなくなります。- 相関ではなく絶対誤差を使うのは、この分野で必要なのが「順位が合っていること」ではなく  **測定値そのものが専門家の計測と一致すること**だからです（研究室間で比較可能な標準を作るのが目的）。**このnotebookの手法が指標をどう最適化しているか**:- 指標が **tolerance正規化MAE** なので、「全体的にわずかに偏っている（バイアスがある）」状態を直すのが最も効率的です。  実際このnotebookは、**MTに一律 -0.4mm** の補正をかけただけで 0.77681 → 0.76794 と大きく改善しています。  これは典型的な**系統誤差（バイアス）の除去**であり、モデルを強くするより費用対効果が高い。- **`FL·sin(PA) ≥ MT`** という関係は、羽状筋の幾何から導かれる恒等式に近い制約です  （筋束が角度PAで斜めに走るとき、その垂直成分が筋厚に相当する）。  これを**予測値の整合性チェック**として使い、整合しない行だけ補正の適用可否を切り替えています。  ドメイン知識を「損失関数」ではなく「**後処理のゲート**」として入れる方法の実例です。

# [LB 0.76704] No-Train Anatomy-Calibrated DLTrack

This notebook releases our strongest UMUD submission and the reasoning path
behind it so that anyone in the competition can fork it and continue.

**Known public LB: 0.76704**  
**Training: none**  
**Accelerator: none (CPU only)**  
**Runtime: seconds, not minutes**

The notebook embeds only a scored prediction anchor—not competition images or
labels—and deterministically rebuilds `submission.csv`. It fails closed unless
the output has exactly 309 rows, the competition schema/ID order, and the
SHA-256 of the scored file.

## Score path and what actually worked

| Step | Public LB | Lesson |
|---|---:|---|
| AmbrosM historical output | 1.33851 | A useful geometric baseline, but not enough |
| Exact DLTrack v0.3.1 recovery | 0.77866 | Correct model lineage and five-frame reduction mattered |
| Agreement-gated MT | 0.77721 | MT was the reliable remaining lever |
| Sequence MT extrapolation 1.5x | 0.77681 | Small bounded continuation helped; 2.0x regressed |
| Global MT -0.4 mm (C130) | 0.76794 | External benchmark transferred the MT bias direction |
| Value-aware proportional envelope (C134) | 0.76728 | Relaxing low MT upward beat a uniform-only rule |
| Lower-tail half-step (C135) | 0.76727 | Essentially saturated |
| Anatomy-gated upper branch (C136) | 0.76711 | `FL*sin(PA)` was useful as a Boolean consistency vote |
| LOIO-stopped continuation (C137) | **0.76704** | Final scored release |

Negative evidence matters too: a broad FL correction scored 0.82498, an
extra-negative scale/device MT rule scored 0.76868, and wider continuation
without a stopping rule was not promoted.

## Method in one picture

`DLTrack predictions -> five-frame sequence reduction -> global MT calibration
-> value-aware MT envelope -> anatomy consistency gate -> submission`

The final stage uses:

- a pooled mean-rater proportional factor `q_mean`;
- a lower-tail gate inherited from the 35-image seven-rater benchmark;
- a more robust cross-lineage median-rater factor;
- `FL*sin(PA) >= pre-global MT` only as a Boolean anatomy gate;
- a leave-one-image-out stopped half continuation on the same 59 rows.

No row is selected from hidden leaderboard residuals. Public LB feedback was
used only to accept or reject predeclared mechanisms.

## License and reuse

Notebook code is released under **GPL-3.0**. Competition data remains governed
by the original competition rules and CC BY-NC-SA terms. Fork this notebook,
replace the prediction anchor with a better measurement pipeline, and keep the
same schema/hash/activation checks. Public LB is not a guarantee of private
leaderboard performance.


### セル1: 決定論的な提出ファイル再構築ロジック（What / Why）**何をしているか**: この大きなセルは、次を行う関数群を定義します。1. `_decimal()` — 文字列を `Decimal` に変換し、非有限値（inf/nan）を拒否する2. `_format_decimal()` — 出力を「偶数丸め（ROUND_HALF_EVEN）」で15桁に量子化して文字列化する3. `_anatomical_mt(pa, fl)` — `FL × sin(PA)` を計算する（解剖学的に期待されるMT）4. `_load_anchor()` — 埋め込みCSVの **SHA-256・列名・行数(309)・image_idの一意性** を全部検証する5. `build_submission_from_bytes()` — C134→C135→C136→C137 の補正チェーンを順に適用して最終CSVを書く補正チェーンの中身は、`pre_global = c130_mt + 0.4`（一律オフセット）を起点に、- **C134**: `max(元のMT, pre_global × q_mean)` — 加法補正と比例補正の**包絡（大きいほうを採る）**- **C135**: MTが `17.038mm` 以下の下側だけ、C134の変化量の**半分をもう一歩追加**- **C136**: `FL·sin(PA) ≥ pre_global` が成り立つ行だけ上側の分岐を有効化（**解剖学的整合ゲート**）- **C137**: leave-one-image-out で止めどころを決めた**有界な継続****なぜそうするのか（設計として学ぶべき点）**:- **`float` ではなく `Decimal(prec=60)` を使う理由**: このnotebookの目的は  「**採点済みのCSVをビット単位で再現すること**」です。`float` は2進数なので `0.4` を正確に表現できず、  演算順序が変わると最下位桁がずれます。CSVの文字列表現が1文字でも変われば SHA-256 が変わり、  「同じ提出のはずなのに再現できない」状態になる。**正確な十進演算が要件のとき `Decimal` を使う**、  という判断は金額計算などでも同じです。- **`ROUND_HALF_EVEN`（銀行家丸め）**: 0.5をつねに切り上げると、丸め誤差が正方向に蓄積します。  偶数側に丸めることで、多数の行にわたって**平均的なバイアスが打ち消される**。MAEを競う場面では地味に効きます。- **フェイルクローズド設計**: `_load_anchor()` はハッシュ・列・行数・ID一意性を全部チェックし、  1つでも外れたら例外を投げます。「入力が想定と違うのに、それらしい出力を出してしまう」ことが  一番危険——という思想です。**壊れるなら黙ってではなく、大きな音を立てて壊れるべき**。- **`max(加法補正, 比例補正)` という包絡**: 「どちらの補正が正しいか分からないので、**保守的なほう（＝元の値から遠ざかりすぎないほう）**を採る」  という考え方です。2つのもっともらしい仮説があるとき、片方に賭けずに**両方と矛盾しない範囲**を採るのは、  検証データが少ないときの合理的な戦略です。> 用語: **キャリブレーション（校正）** = モデルの出力を、真値の分布に合うように後から補正すること。> 分類なら確率の校正（Platt scaling / isotonic）、回帰ならバイアス補正やスケール補正が該当します。> **モデルを触らずに指標を改善できる**ため、コンペ終盤の定番手段です。> ⚠️ **鵜呑みにしない**: `q_mean = 0.98045972631703322153145091365007130743938770034627434305509` のように> 60桁の定数がハードコードされていますが、**309行のテストデータに対して有効数字60桁の意味はありません**。> これは「採点済みCSVをビット再現するため」の実装上の都合であって、> **その精度で校正係数が推定できているという意味ではない**。桁数の多さを精度の高さと誤解しないこと。

In [ ]:
"""Self-contained reproduction of the public-LB 0.76704 UMUD submission.

The notebook release embeds the scored C130 prediction anchor, then applies
the deterministic C134-C137 calibration path.  No training, network access,
competition images, hidden labels, or third-party packages are required.
"""

from __future__ import annotations

import csv
import hashlib
import io
import math
from decimal import Decimal, ROUND_HALF_EVEN, localcontext
from pathlib import Path
from typing import Sequence

SUBMISSION_COLUMNS = ("image_id", "pa_deg", "fl_mm", "mt_mm")
EXPECTED_ROWS = 309
EXPECTED_ANCHOR_SHA256 = (
    "28684ca9a898c1df0697bca8bf6d3d60d61789e76e373355fb25075ba1860d03"
)
EXPECTED_OUTPUT_SHA256 = (
    "c8100f479a3bbbd2956451c94a68382d1062b295fcfeacb39908943560b252c8"
)

GLOBAL_MT_CORRECTION_MM = Decimal("0.4")
Q_MEAN = Decimal(
    "0.98045972631703322153145091365007130743938770034627434305509"
)
LOW_TAIL_GATE_MM = Decimal("17.038237766946587")
Q_ROBUST = Decimal(
    "0.981508489028500734911029995862899219380735533755795380367775"
)
Q_FINAL = Decimal(
    "0.982032870384234491600819536969313175351409450460555899024118"
)
HALF_STEP = Decimal("0.5")
OUTPUT_QUANTUM = Decimal("0.000000000000001")


def _decimal(value: str, *, field: str, image_id: str) -> Decimal:
    try:
        number = Decimal(value.strip())
    except Exception as exc:
        raise ValueError(f"{image_id}: invalid {field}={value!r}") from exc
    if not number.is_finite():
        raise ValueError(f"{image_id}: non-finite {field}")
    return number


def _format_decimal(value: Decimal) -> str:
    rounded = value.quantize(OUTPUT_QUANTUM, rounding=ROUND_HALF_EVEN)
    text = format(rounded, "f").rstrip("0").rstrip(".")
    return text if "." in text else f"{text}.0"


def _anatomical_mt(pa_deg: Decimal, fl_mm: Decimal) -> Decimal:
    value = float(fl_mm) * math.sin(math.radians(float(pa_deg)))
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"invalid anatomical geometry: PA={pa_deg}, FL={fl_mm}")
    return Decimal(str(value))


def _load_anchor(anchor_bytes: bytes) -> list[dict[str, str]]:
    digest = hashlib.sha256(anchor_bytes).hexdigest()
    if digest != EXPECTED_ANCHOR_SHA256:
        raise ValueError(
            f"C130 anchor SHA-256 mismatch: expected {EXPECTED_ANCHOR_SHA256}, got {digest}"
        )
    reader = csv.DictReader(io.StringIO(anchor_bytes.decode("utf-8")))
    if tuple(reader.fieldnames or ()) != SUBMISSION_COLUMNS:
        raise ValueError(f"unexpected anchor columns: {reader.fieldnames}")
    rows = list(reader)
    if len(rows) != EXPECTED_ROWS:
        raise ValueError(f"expected {EXPECTED_ROWS} anchor rows, got {len(rows)}")
    ids = [row["image_id"] for row in rows]
    if len(set(ids)) != EXPECTED_ROWS:
        raise ValueError("anchor image_id values are not unique")
    for row in rows:
        for field in SUBMISSION_COLUMNS[1:]:
            _decimal(row[field], field=field, image_id=row["image_id"])
    return rows


def build_submission_from_bytes(
    anchor_bytes: bytes,
    output_path: str | Path,
    *,
    sample_ids: Sequence[str] | None = None,
) -> dict[str, object]:
    """Rebuild the exact scored C137 CSV from the frozen C130 anchor."""

    rows = _load_anchor(anchor_bytes)
    anchor_ids = [row["image_id"] for row in rows]
    if sample_ids is not None and anchor_ids != list(sample_ids):
        raise ValueError("anchor IDs/order do not match the competition sample submission")

    output_rows: list[dict[str, str]] = []
    c134_changed = low_tail_changed = anatomy_changed = 0

    with localcontext() as context:
        context.prec = 60
        for row in rows:
            image_id = row["image_id"]
            c130_mt = _decimal(row["mt_mm"], field="mt_mm", image_id=image_id)
            pa_deg = _decimal(row["pa_deg"], field="pa_deg", image_id=image_id)
            fl_mm = _decimal(row["fl_mm"], field="fl_mm", image_id=image_id)
            pre_global = c130_mt + GLOBAL_MT_CORRECTION_MM

            # C134: conservative envelope of additive and proportional MT calibration.
            c134_continuous = max(c130_mt, pre_global * Q_MEAN)
            c134_quantized = Decimal(_format_decimal(c134_continuous))
            c134_text = (
                _format_decimal(c134_continuous)
                if c134_quantized != c130_mt
                else row["mt_mm"]
            )
            c134 = Decimal(c134_text)
            if c134 != c130_mt:
                c134_changed += 1

            # C135: one bounded half-step on the lower half of the active branch.
            c135_text = c134_text
            if c134 > c130_mt and pre_global <= LOW_TAIL_GATE_MM:
                c135_text = _format_decimal(c134 + HALF_STEP * (c134 - c130_mt))
                low_tail_changed += 1
            c135 = Decimal(c135_text)

            # Reconstruct the continuous parent used by the anatomy gate.
            c135_continuous = c134_continuous
            if c134_continuous > c130_mt and pre_global <= LOW_TAIL_GATE_MM:
                c135_continuous = c134_continuous + HALF_STEP * (
                    c134_continuous - c130_mt
                )

            active = pre_global * Q_MEAN > c130_mt
            robust_proposal = pre_global * Q_ROBUST
            anatomy_margin = _anatomical_mt(pa_deg, fl_mm) - pre_global
            anatomy_eligible = (
                active
                and pre_global > LOW_TAIL_GATE_MM
                and robust_proposal > c135_continuous
                and anatomy_margin >= 0
            )

            final_text = c135_text
            if anatomy_eligible:
                final_text = _format_decimal(pre_global * Q_FINAL)
                anatomy_changed += 1

            output_rows.append(
                {
                    "image_id": image_id,
                    "pa_deg": row["pa_deg"],
                    "fl_mm": row["fl_mm"],
                    "mt_mm": final_text,
                }
            )

    if (c134_changed, low_tail_changed, anatomy_changed) != (174, 30, 59):
        raise ValueError(
            "unexpected activation counts: "
            f"C134={c134_changed}, C135={low_tail_changed}, anatomy={anatomy_changed}"
        )

    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=SUBMISSION_COLUMNS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(output_rows)
    output_bytes = buffer.getvalue().encode("utf-8")
    output_sha256 = hashlib.sha256(output_bytes).hexdigest()
    if output_sha256 != EXPECTED_OUTPUT_SHA256:
        raise ValueError(
            f"output SHA-256 mismatch: expected {EXPECTED_OUTPUT_SHA256}, got {output_sha256}"
        )

    destination = Path(output_path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(output_bytes)
    return {
        "output": str(destination),
        "rows": len(output_rows),
        "columns": list(SUBMISSION_COLUMNS),
        "c134_changed_rows": c134_changed,
        "c135_low_tail_changed_rows": low_tail_changed,
        "anatomy_gated_rows": anatomy_changed,
        "sha256": output_sha256,
        "known_public_lb": 0.76704,
    }


### セル2: 埋め込みアンカーの展開と実行（What / Why）**何をしているか**: base64でエンコードされgzip圧縮された文字列 `C130_ANCHOR_B64` をデコード・展開して、採点済みの予測CSVのバイト列を復元し、前のセルで定義した `build_submission_from_bytes()` に渡して`submission.csv` を書き出します。**なぜそうするのか**:- **なぜソースコードに埋め込むのか**: Kaggleのnotebookは外部ネットワークにアクセスできない設定で  実行されることが多く、また別データセットに依存すると**そのデータセットが更新・削除されると壊れます**。  ソースに埋め込めば、notebook単体で完全に自己完結し、フォークした人が必ず同じ結果を得られます。  「1ファイルで完結し、外部依存ゼロで再現できる」というのは配布物として非常に強い性質です。- **base64 + gzip の理由**: CSVはテキストなのでgzipがよく効き（数分の1に縮む）、  base64はバイナリを「ソースコードに書ける安全な文字」だけに変換する標準手段です。  この組み合わせは、設定・小さな辞書・学習済みの小さな重みなどを配布するときの定番パターンです。**ここで立ち止まって批判的に考えるべきこと**:このnotebookの中核は「学習済みモデル」ではなく「**採点済みの答案そのもの**」です。つまり:1. **新しい画像には一切適用できません。** テスト309行のIDに紐付いた値しか持っていないので、   別の超音波画像が来たら何も出力できない。これは**モデルではなくルックアップテーブル**です。2. **補正パラメータはPublic LBのフィードバックで選ばれています。** notebook自身が   「Public LB feedback was used only to accept or reject predeclared mechanisms（事前に宣言した機構の採否にのみ使った）」   と書いており、原著者は自覚的かつ誠実です。しかしそれでも、**Public LBという309行の一部に対して   10回以上の採否判断を重ねた**という事実は変わりません。これは統計的には**多重比較**であり、   Private LBでは効果が縮む（あるいは反転する）と考えるのが自然です。   原著者自身、最後に「**Public LB is not a guarantee of private leaderboard performance**」と明記しています。3. **一方で、記録の仕方は見習うべきです。** 「どの機構が何点動かしたか」の表に加えて、   `a broad FL correction scored 0.82498`（大幅悪化）のような**否定的な結果も併記**しています。   効かなかった実験を残すことは、後続が同じ失敗を繰り返さないために極めて価値が高い。   自分の実験ログでも「試して駄目だったこと」を必ず書き残しましょう。

In [ ]:
# The compressed payload is the scored C130 prediction anchor.
import base64
import gzip
import json
from pathlib import Path

C130_ANCHOR_B64 = """H4sIAAAAAAACA62bW4okORJF/2ctjiN7m61gmI9eQ9HQDxqmhv7o/TNXUZGKbCoaSu6WkPXIuHYSXCbTNUn+x9eff//1yx+/HH/+/OWXX38/fvvvl69fj69/4c9//eenf38Z+KLzrz9+OyjOQcQRlWFUXHI4ncIH5TlqUJBKcpKSr0i+HCmXI/VypF2O9I9IZS8JLTUiGXG4nDlclZXCM0MPqpN5hIuSeQqVLkq0ULKFUh0UGi2UZ/7lqYMMj99J2FmPrDOcI8UHPiHjCYkgJ6uRI/iF4PsIuY/Q+wi7j1jJKv7KbHYJUSu3VGFb4tgR5464NsQ8dsT0SRxxpllFGluKEB3MZ7KK5DDFLK5ccXwxTi7G6cU4uxj3/cDjuQqNT1+8xLEjzh1xbYhl7IjXwHNUqHiUsIyVMXhUgprOA88KhWiVb+GLcXIxTi/G2cW4NzO+UJIFU8WSA7lCSxsb2tzQ1o9rdWxo6XttnCHBYVhg2ENszQDlHbHsiHVHbDviN4Nnp7pTxjDkRMWoJY4dce6Ia0NsY0f8ZgT9HG4xkOOKZA5bi5fxjlh2xLojth2xf6zZbGR4kgkd5ZCDYG0qKXQEwW6Wo7rBZDrxKLOYf69Et2ihZAulOig+Wijf0geLIjw7jABiOJPYj4RJKisq+MxC7h1YAFEck5I5A98uC8IdEOmAaAfEOiDPtLVT4CPgKBWOguwIwQ9I0SA5EOazNptqUGFgdHi+rKbHfUTeR9RtRIz7iGeZ4zOQ6jxTezaYWL0DQ6AEQ5wGL5/pj7qShVVAsGRZZK5BCW6hSAtFWyjWQnmzSioyuzgVMEGjkK+hiB1x7ohrQ5xjR/xMHzlRAQ1PAWlImLB1ICVQNdFdo17aFM6KajQyCHMYNoZWZ5TcAZEOiHZArAPyLXPQVcwZO761TZqpB2YsKdYgLThS16hZUKXU0GghJzl5mbSMDkh2QKoBUqMDsrZtxPFzYYwENHxg2LwIrWrOxp0lJgO/AhO7apAr5vqCcAdEOiDaAbEOiH+s7qiBVlxw06LKfGCZqppcFUZrGHowFigmDxSbzHB0TosSLZRsoVQDhcZooTz9JJ+owCJw714oJH4gnxVjkcDOjafMg+PEf0Od4bDKknRBuAMiHRDtgFgH5LlE65niSH+VxMcOtz/yJHNNwRpfmGg53T4GEFk/q7njI1uUaKFkC6U6KB+b5Dcp9OFQ4eoxFIoyYj5HBEstpimxoLQ4D9U5joXl0kWL3Aqt16JwC0VaKNpCsRbKauFhR5UwGjy9xLSoLo991rJgNFZzzRzsUXMXAB0Vys5iRAMjGxh1n8GjgfHarYUcg4IkR6uKrgr2Qxz/hctHT5poBRi+OLnGY2rgX77K9WvT/hZFWijaQrEWymqIYp4QYTwYZkoNi2HNgxOfx2wYNdNH00o294bgPGDqBsmiRAslWyjVQXkdKNyiPHN3HrrBtBWchRkJjBxslcGyoYwEk7o8VsRHc2KOLgWt3WuMPs4dblKkhaItFGuhPA0uTZc8T7UK0wBOeLpk2DlUUkUTAiP3bQI89gfmwacYivaCRAckOyDVAHmeidyEPNN2bqygoUPjASs3IdOcwRazFlo71Jd4Wg0Ln9Xl0f7FonALRVoo2kKxFsoyuPBmeOyVswlWmU0z3JtboPWAeSubCyd+CWp2wN/p3A5ZkOiAZAekGiA2OiCvva/wmdmP3YXyPEIRExqozq6R0zT7CR78MXo7kQBrUbiFIi0UbaFYC8U/KBZI8sGCCl1s9LhbIHM/FkXEajyGSOD2fI5O6jBajGhgZAOj7jM+jqVuMegj7YMcZTgeixwK9pHIe6R4wnOEO5bCOTKzSJdiGgicxTo4pOep1F2KtFC0hWItlNWNCRaAkprHAeijc270OExzEToNNJBes6eTgf4kczBaPV1p79EByQ5INUBidEDWJgIPDiz06IvnppnOjZ7HSRpcBZzVYx00TAZMC9QXtHf8qVwHt1CkhaItFGuhLFsLoQeSGoOR6BoP8BNFhdJiHm7xY4jgN5D0805XoO9fkOiAZAekGiA5OiCfLijn7JeNleY9uTk8MeaNAUFJ5mfXwZGMfjrLQ/K1g5bcQpEWirZQrIXiH+a4MCmN0WIYPAXngU90brvPfQKqwE8YokRhQUeigqmyGubnAdlNSHZAqgHyPCC7CVlp64LwcjTBFhXzLq+jt0NvgYUQU+AQOpHuJjJ1rOv2GxXfR8h9hN5H2H3Euq/iAxkclmYo1GiIY94YRTunUu5oOh6nwTyPK+cRkCbM84JEByQ7IHUfwmN0QOjH7+Dy4B2x7Ih1R2w74uUnNZBTmLd4RGoSB5ymOfolePvBqO3xyNDy2Z2Gq+l4PaXogGQHpBogNDogO5lDO5lDO5lDO5lDO5lDviOOHXHuiDeu1DOPHfHa6C7P6dBCC3UZbQX6J5J5cI62baDjeBQQFA4nhWWDkYjVVzBzC0VaKNpCsRbKTvrsvJHBO29k8M4bGbzzRgbL2nB+HLpjCoTObenXrXdGbzrfbJj2aL3yxcIX4+RinF6Ms4txOwMvOwMvOwMvOwOvOwP/99cy+JwHFGg2dG6YD30cY8yLjpgW82JIWKxAvhooVwP1aqBdDfS/BxrNs+Oi2UbA9j4aNq8IEtQKn/c8VmRcjszLkXU18u+vgWxFvn2rR2d36kPdiFxWrhmff/7v9x8Vy45Yd8S2I/bvxVhL/kEcO+LcEdeG2MeOmN6JST59rdF23tDKhlY3tLah/X7w/rkSeuyIc0dcG+IYO2J6l8qFvvA11rbEvCOWHbHuiG1HvEbQBj60SMrp0A7UEvg5UvyU4OfmxXF77KhzQIcaZesSCEd0QLIDUg2QHB2QN5kzD8kKhTeE5gXcXGmWvCOWHbHuiG1H/MycOm1e1+Rp+gzfn1Luc82QtehnXA3Mq4F1MbDG1cC3RX8el0pQzfdPfE3B4g2tbGh1Q2sb2jcr9nzl89PX6znEhjY3tPXDWhljQ/t21tq7xU8Gb2hlQ6sbWtvQ+rt1ATkt82KJzc6dXw8idsS5I64fEf8fUo507o5HAAA="""
anchor_bytes = gzip.decompress(base64.b64decode(C130_ANCHOR_B64))

competition_dir = Path("/kaggle/input/umud-challenge-muscle-architecture-in-ultrasound-data")
sample_path = competition_dir / "sample_submission.csv"
sample_ids = None
if sample_path.exists():
    sample_text = sample_path.read_text(encoding="utf-8-sig")
    first_line = sample_text.splitlines()[0]
    delimiter = ";" if first_line.count(";") > first_line.count(",") else ","
    template_ids = [
        row["image_id"]
        for row in csv.DictReader(sample_text.splitlines(), delimiter=delimiter)
    ]
    if len(template_ids) == EXPECTED_ROWS:
        sample_ids = template_ids
        print(f"Validated against {sample_path}")
    else:
        print(
            f"Official sample is a {len(template_ids)}-row format example, "
            "not the 309-row manifest; using frozen ID/order/hash checks."
        )
else:
    print("Competition sample not mounted; using frozen schema/ID/hash checks.")

working_dir = Path("/kaggle/working")
if not working_dir.exists():
    working_dir = Path.cwd()
output_path = working_dir / "submission.csv"
summary = build_submission_from_bytes(
    anchor_bytes,
    output_path,
    sample_ids=sample_ids,
)
print(json.dumps(summary, indent=2))
print()
print("First five rows:")
print("".join(output_path.read_text(encoding="utf-8").splitlines(True)[:6]))


## Good directions for the next team

1. Replace the frozen C130 anchor with a stronger open segmentation model while
   preserving five-frame grouping and explicit failure handling.
2. Validate anatomy gates on genuinely paired human labels; do not tune row IDs
   from scalar leaderboard scores.
3. Treat PA, FL, and MT as related measurements, but remember that
   `FL*sin(PA)` is only an approximation when fascicles curve or aponeuroses are
   not parallel.
4. Keep experiments causal: one bounded mechanism, one score, then adapt.

If you improve it, publish the mechanism—not just the CSV. Good luck. 🫡
